# Разведочный анализ данных (exploratory data analysis)

## Импортирование библиотек

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

## Загрузка датасета

In [ ]:
PROCESSED_PATH = Path("../data/processed/cpi_monthly.csv")
df = pd.read_csv(PROCESSED_PATH, parse_dates=["Date"])

print(df.shape)
print(df.dtypes)

## Базовая проверка данных

In [ ]:
cols = df.columns
colours = ['#000099', '#ffff00']
sns.heatmap(df[cols].isnull(), cmap=sns.color_palette(colours))
plt.title("Карта отсутствующих данных")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(df['cpi_mom'], bins=50, color='#3B6E9F', edgecolor='black')

ax.set_xlabel('cpi_mom, %', fontsize=12)
ax.set_ylabel('Частота', fontsize=12)
ax.set_title('Распределение данных и их частота', fontsize=14)

ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
df.boxplot(column=['cpi_mom'])
plt.title("Коробчатая диаграмма cpi_mom")
plt.show()

In [ ]:
s = df['cpi_mom']
n = len(s)

mean_val = s.mean()
stderr_val = s.std(ddof=1) / np.sqrt(n)
median_val = s.median()
mode_val = s.mode().iloc[0]
std_val = s.std(ddof=1)
var_val = s.var(ddof=1)
kurt_val = stats.kurtosis(s, fisher=True)
skew_val = stats.skew(s)
range_val = s.max() - s.min()
min_val = s.min()
max_val = s.max()
sum_val = s.sum()
count_val = n
conf_val = stats.t.ppf(0.975, df=n - 1) * stderr_val

article = {
    'Среднее': 103.68,
    'Стандартная ошибка': 0.73,
    'Медиана': 100.8,
    'Мода': 100.4,
    'Стандартное отклонение': 14.13,
    'Дисперсия выборки': 199.71,
    'Эксцесс': 230.4,
    'Асимметричность': 13.84,
    'Интервал': 245.8,
    'Минимум': 99.5,
    'Максимум': 345.3,
    'Сумма': 38881.45,
    'Счёт': 375,
    'Уровень надёжности (95,0%)': 1.434954084,
}

ours = {
    'Среднее': mean_val,
    'Стандартная ошибка': stderr_val,
    'Медиана': median_val,
    'Мода': mode_val,
    'Стандартное отклонение': std_val,
    'Дисперсия выборки': var_val,
    'Эксцесс': kurt_val,
    'Асимметричность': skew_val,
    'Интервал': range_val,
    'Минимум': min_val,
    'Максимум': max_val,
    'Сумма': sum_val,
    'Счёт': count_val,
    'Уровень надёжности (95,0%)': conf_val,
}

table1 = pd.DataFrame({
    'Показатель': list(ours.keys()),
    'Статья': [article[k] for k in ours.keys()],
    'Полученные данные': [round(v, 4) for v in ours.values()],
})
table1['Разница со статьёй'] = (table1['Полученные данные'] - table1['Статья']).round(4)
display(table1)

In [ ]:
df['Date'].dt.year.value_counts().sort_index().plot.bar(figsize=(14, 4))
plt.title("Число наблюдений по годам")
plt.show()

In [ ]:
print("Первые 3 строки (январь–март 1991):")
display(df.head(3))

print("\nПоследние 3 строки (январь–март 2022):")
display(df.tail(3))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['Date'], df['cpi_mom'], linewidth=1, color='steelblue')
ax.axhline(100, color='red', linestyle='--', linewidth=0.8, label='100')
ax.set_title('Индекс потребительских цен (ИПЦ) в России (месяц к месяцу) с января 1991 по март 2022')
ax.set_xlabel('Год')
ax.set_ylabel('ИПЦ, %')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
split_date = pd.Timestamp('2022-01-01')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['Date'], df['cpi_mom'], linewidth=1, color='steelblue', label='ИПЦ')
ax.axvline(split_date, color='red', linestyle='--', linewidth=1.2, label='Отсечка')
ax.axhline(100, color='gray', linestyle=':', linewidth=0.8)

train_part = df[df['Date'] < split_date]
test_part = df[df['Date'] >= split_date]
ax.scatter(train_part['Date'], train_part['cpi_mom'], s=6, color='steelblue', alpha=0.5, label='Запись тренировочной выборки')
ax.scatter(test_part['Date'], test_part['cpi_mom'], s=20, color='red', label='Запись тестовой выборки')

ax.xaxis.set_major_locator(mdates.YearLocator(base=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[1, 7]))
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

ax.set_title('Индекс потребительских цен (ИПЦ) в России (месяц к месяцу)\nс января 1991 по март 2022 с отсечкой в январе 2022')
ax.set_xlabel('Год')
ax.set_ylabel('ИПЦ, %')

ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
fig.subplots_adjust(right=0.75)
plt.show()

print(f"train R2: {len(train_part)} наблюдений ({train_part['Date'].min().date()} — {train_part['Date'].max().date()})")
print(f"test  R2: {len(test_part)} наблюдений ({test_part['Date'].min().date()} — {test_part['Date'].max().date()})")